# Python PEP Analysis
## A timeline and influence study of Python Enhancement Proposals

This notebook:
1. **Fetches** all PEPs from the official `python/peps` GitHub repository
2. **Visualizes** a timeline of PEPs by type, status, and era
3. **Scores influence** via cross-references, landmark curation, and external signals
4. **Narrates Python's evolution** — how the language changed decade by decade

In [2]:
import re
import time
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

print("Dependencies loaded.")

Dependencies loaded.


## 1. Fetch PEP Data from GitHub

We use the GitHub API to list all `.rst` files in the `python/peps` repo, then download and parse each one.

In [ ]:
import os

GITHUB_API = "https://api.github.com"
PEPS_REPO = "python/peps"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

session = requests.Session()
if GITHUB_TOKEN:
    session.headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    print("GitHub token loaded.")
else:
    print("No token found — you may hit rate limits.")
session.headers["Accept"] = "application/vnd.github.v3+json"

def github_get(url, **params):
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def list_pep_files():
    """Return list of {name, download_url} for all pep-NNNN.rst files."""
    print("Fetching file list...")
    items = github_get(f"{GITHUB_API}/repos/{PEPS_REPO}/contents/peps")
    results = [
        {"name": i["name"], "download_url": i["download_url"]}
        for i in items
        if re.match(r'^pep-\d{4}\.rst$', i["name"])
    ]
    print(f"  Found {len(results)} PEP files.")
    return results

pep_files = list_pep_files()

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADER_FIELDS = {"pep", "title", "author", "status", "type", "created", "python-version", "superseded-by", "replaces"}

def parse_pep(name, text):
    number = int(re.search(r'\d+', name).group())
    headers = {}
    body_lines = []
    in_header = True
    current_key = None

    for line in text.splitlines():
        if in_header:
            if line.strip() == "" and headers:
                in_header = False
                continue
            m = re.match(r'^([A-Za-z][A-Za-z -]+?):\s*(.*)', line)
            if m:
                current_key = m.group(1).lower().strip()
                headers[current_key] = m.group(2).strip()
            elif line.startswith(" ") and current_key:
                headers[current_key] = headers[current_key] + " " + line.strip()
        else:
            body_lines.append(line)

    body = "\n".join(body_lines)

    refs = set(int(n) for n in re.findall(r'(?i)(?::pep:`(\d+)`|pep\s+(\d+))', body)
               for n in filter(None, [n[0] or n[1]]))
    refs.discard(number)

    def parse_date(s):
        for fmt in ("%d-%b-%Y", "%d %b %Y", "%Y-%m-%d", "%b %Y"):
            try:
                return datetime.strptime(s.strip(), fmt)
            except ValueError:
                pass
        return None

    created = parse_date(headers.get("created", ""))
    return {
        "number": number,
        "title": headers.get("title", ""),
        "author": headers.get("author", ""),
        "status": headers.get("status", "Unknown"),
        "type": headers.get("type", "Unknown"),
        "created": created,
        "python_version": headers.get("python-version", ""),
        "superseded_by": headers.get("superseded-by", ""),
        "replaces": headers.get("replaces", ""),
        "refs": refs,
        "body": body,
    }

def fetch_one(f):
    r = session.get(f["download_url"], timeout=30)
    r.raise_for_status()
    return parse_pep(f["name"], r.text)

def fetch_all_peps(pep_files, max_workers=20):
    records = []
    failed = []
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_one, f): f for f in pep_files}
        for i, future in enumerate(as_completed(futures), 1):
            try:
                records.append(future.result())
            except Exception as e:
                failed.append((futures[future]["name"], str(e)))
            if i % 50 == 0 or i == len(pep_files):
                print(f"  [{i}/{len(pep_files)}] {time.time() - start:.0f}s elapsed")
    if failed:
        print(f"  Warning: {len(failed)} PEPs failed to fetch: {[n for n, _ in failed]}")
    return records

print(f"Fetching {len(pep_files)} PEPs concurrently...")
start = time.time()
raw_peps = fetch_all_peps(pep_files)
print(f"Done — {len(raw_peps)} PEPs fetched in {time.time() - start:.1f}s")

In [ ]:
import json
from pathlib import Path

CACHE_FILE = Path("peps_cache.json")
CACHE_MAX_AGE_DAYS = 7  # re-fetch if cache is older than this
FORCE_REFRESH = False   # set to True to bypass cache regardless of age

def save_peps(records):
    serializable = [
        {**r, "created": r["created"].isoformat() if r["created"] else None, "refs": list(r["refs"])}
        for r in records
    ]
    payload = {"fetched_at": datetime.utcnow().isoformat(), "peps": serializable}
    CACHE_FILE.write_text(json.dumps(payload, indent=2))
    print(f"Saved {len(records)} PEPs to {CACHE_FILE}")

def load_peps():
    if not CACHE_FILE.exists():
        return None
    payload = json.loads(CACHE_FILE.read_text())
    fetched_at = datetime.fromisoformat(payload["fetched_at"])
    age_days = (datetime.utcnow() - fetched_at).days
    if age_days > CACHE_MAX_AGE_DAYS:
        print(f"Cache is {age_days} days old (max {CACHE_MAX_AGE_DAYS}) — refreshing.")
        return None
    records = payload["peps"]
    for r in records:
        r["created"] = datetime.fromisoformat(r["created"]) if r["created"] else None
        r["refs"] = set(r["refs"])
    print(f"Loaded {len(records)} PEPs from cache (fetched {age_days}d ago).")
    return records

if not FORCE_REFRESH:
    cached = load_peps()
else:
    print("FORCE_REFRESH=True — skipping cache.")
    cached = None

if cached is not None:
    raw_peps = cached
else:
    raw_peps = fetch_all_peps(pep_files)
    save_peps(raw_peps)

In [ ]:
# Build DataFrame and compute cross-reference counts
df = pd.DataFrame(raw_peps)
df = df[df["created"].notna()].copy()
df["year"] = df["created"].dt.year

# How many other PEPs reference each PEP?
all_numbers = set(df["number"])
ref_counts = {n: 0 for n in all_numbers}
for _, row in df.iterrows():
    for ref in row["refs"]:
        if ref in ref_counts:
            ref_counts[ref] += 1

df["cited_by_count"] = df["number"].map(ref_counts)

# Normalize status labels
STATUS_MAP = {
    "Final": "Final", "Active": "Active", "Accepted": "Accepted",
    "Draft": "Draft", "Deferred": "Deferred",
    "Rejected": "Rejected", "Withdrawn": "Withdrawn",
    "Superseded": "Superseded",
}
df["status_clean"] = df["status"].map(lambda s: STATUS_MAP.get(s, "Other"))

# Assign era
def assign_era(year):
    if year < 2000: return "Pre-2000"
    if year < 2006: return "Python 2 peak (2000–05)"
    if year < 2012: return "Py3 transition (2006–11)"
    if year < 2016: return "Stabilization (2012–15)"
    if year < 2020: return "Type hints era (2016–19)"
    return "Modern Python (2020+)"

df["era"] = df["year"].map(assign_era)

print(df[["number","title","status_clean","type","year","cited_by_count"]].sort_values("cited_by_count", ascending=False).head(10))

## 2. PEP Timeline

Each dot is a PEP positioned by creation date. Color = type, shape encodes status. Hover for details.

In [ ]:
TYPE_COLORS = {
    "Standards Track": "#4C72B0",
    "Informational": "#55A868",
    "Process": "#C44E52",
    "Unknown": "#8c8c8c",
}

STATUS_SYMBOLS = {
    "Final": "circle",
    "Active": "star",
    "Accepted": "diamond",
    "Draft": "circle-open",
    "Rejected": "x",
    "Withdrawn": "cross",
    "Superseded": "triangle-down",
    "Deferred": "square-open",
    "Other": "circle-open",
}

# Jitter y-axis by type so points don't stack
TYPE_Y = {"Standards Track": 1, "Informational": 2, "Process": 3, "Unknown": 0}
df["y_pos"] = df["type"].map(lambda t: TYPE_Y.get(t, 0))

fig = go.Figure()

for ptype, group in df.groupby("type"):
    for status, sub in group.groupby("status_clean"):
        fig.add_trace(go.Scatter(
            x=sub["created"],
            y=sub["y_pos"] + (sub["number"] % 7) * 0.12,  # vertical spread within band
            mode="markers",
            name=f"{ptype} / {status}",
            marker=dict(
                color=TYPE_COLORS.get(ptype, "#8c8c8c"),
                symbol=STATUS_SYMBOLS.get(status, "circle-open"),
                size=7,
                opacity=0.75,
                line=dict(width=0.5, color="white"),
            ),
            text=sub.apply(
                lambda r: f"PEP {r['number']}: {r['title']}<br>Status: {r['status_clean']}<br>Cited by: {r['cited_by_count']} PEPs",
                axis=1,
            ),
            hoverinfo="text",
            showlegend=True,
        ))

# Era shading
ERA_SPANS = [
    ("Pre-2000", "1990-01-01", "2000-01-01", "rgba(200,200,200,0.15)"),
    ("Py3 transition\n(2006–11)", "2006-01-01", "2012-01-01", "rgba(100,150,250,0.10)"),
    ("Type hints era\n(2016–19)", "2016-01-01", "2020-01-01", "rgba(250,180,100,0.12)"),
]
for label, x0, x1, color in ERA_SPANS:
    fig.add_vrect(x0=x0, x1=x1, fillcolor=color, line_width=0,
                  annotation_text=label, annotation_position="top left",
                  annotation_font_size=10)

fig.update_layout(
    title="Python PEPs — Timeline by Type and Status",
    xaxis_title="Date Created",
    yaxis=dict(
        tickvals=[0, 1, 2, 3],
        ticktext=["Unknown", "Standards Track", "Informational", "Process"],
        title="PEP Type",
    ),
    height=550,
    legend=dict(orientation="h", yanchor="bottom", y=-0.35),
    hovermode="closest",
    template="plotly_white",
)
fig.show()

In [ ]:
# PEPs per year bar chart, stacked by type
yearly = df.groupby(["year", "type"]).size().reset_index(name="count")

fig2 = px.bar(
    yearly, x="year", y="count", color="type",
    color_discrete_map=TYPE_COLORS,
    title="PEPs Created Per Year by Type",
    labels={"year": "Year", "count": "PEPs Created", "type": "Type"},
    template="plotly_white",
    height=400,
)
fig2.show()

## 3. Influence Scoring

We combine three signals into a composite influence score:

| Signal | Weight | Description |
|---|---|---|
| `cited_by_count` | 0.5 | How many other PEPs reference this one |
| `landmark` | 0.3 | Manually curated landmark PEPs |
| `stackoverflow_proxy` | 0.2 | SO tag popularity for the Python version that introduced the feature |

In [ ]:
# --- Signal 1: Cross-reference count (already computed) ---

# --- Signal 2: Manual landmark list ---
# These PEPs fundamentally shaped Python's direction or community practice
LANDMARK_PEPS = {
    1,    # PEP Purpose and Guidelines
    8,    # Style Guide for Python Code
    20,   # The Zen of Python
    257,  # Docstring Conventions
    302,  # New Import Hooks
    328,  # Imports: Multi-Line and Absolute/Relative
    333,  # Python Web Server Gateway Interface (WSGI)
    343,  # The "with" Statement
    380,  # Syntax for Delegating to a Subgenerator (yield from)
    405,  # Python Virtual Environments
    420,  # Implicit Namespace Packages
    428,  # The pathlib module
    443,  # Flexible function and variable annotations
    484,  # Type Hints
    492,  # Coroutines with async and await
    498,  # Literal String Interpolation (f-strings)
    518,  # Specifying Minimum Build System Requirements (pyproject.toml)
    526,  # Syntax for Variable Annotations
    527,  # Minimizing the Footprint of the CPython Source (PEP index)
    544,  # Protocols: Structural subtyping
    557,  # Data Classes
    560,  # Core support for typing module and generic types
    561,  # Distributing and Packaging Type Information
    572,  # Assignment Expressions (walrus operator)
    585,  # Type Hinting Generics In Standard Collections
    604,  # Allow writing union types as X | Y
    616,  # String methods to remove prefixes and suffixes
    634,  # Structural Pattern Matching
    654,  # Exception Groups and except*
    695,  # Type Parameter Syntax
}

df["is_landmark"] = df["number"].isin(LANDMARK_PEPS).astype(int)

# --- Signal 3: Stack Overflow proxy ---
# Use the SO API to get question counts for python-N.N tags
SO_API = "https://api.stackexchange.com/2.3"

def so_tag_count(tag):
    r = requests.get(
        f"{SO_API}/tags/{tag}/info",
        params={"site": "stackoverflow", "key": ""},
        timeout=15,
    )
    if r.status_code != 200:
        return 0
    items = r.json().get("items", [])
    return items[0]["count"] if items else 0

# Map Python version strings to SO tag counts (cached to avoid many API calls)
print("Fetching Stack Overflow tag counts for Python versions...")
version_tags = ["python-2.7", "python-3.x", "python-3.6", "python-3.7",
                "python-3.8", "python-3.9", "python-3.10", "python-3.11", "python-3.12"]
so_counts = {}
for tag in version_tags:
    count = so_tag_count(tag)
    so_counts[tag] = count
    print(f"  {tag}: {count:,}")
    time.sleep(0.3)

print("Done.")

In [ ]:
def map_so_score(python_version_str):
    """Map a PEP's Python-Version field to a normalized SO question count."""
    if not python_version_str:
        return 0
    # Take the first version listed
    v = python_version_str.split(",")[0].strip()
    # Try exact match first
    tag = f"python-{v}"
    if tag in so_counts:
        return so_counts[tag]
    # Fall back to major.minor prefix
    m = re.match(r'^(\d+\.\d+)', v)
    if m:
        tag = f"python-{m.group(1)}"
        if tag in so_counts:
            return so_counts[tag]
    # Py3 catch-all
    if v.startswith("3"):
        return so_counts.get("python-3.x", 0)
    return 0

df["so_score_raw"] = df["python_version"].map(map_so_score)

# Normalize each signal to [0, 1]
def norm(series):
    mx = series.max()
    return series / mx if mx > 0 else series

df["score_refs"]     = norm(df["cited_by_count"])
df["score_landmark"] = df["is_landmark"].astype(float)
df["score_so"]       = norm(df["so_score_raw"])

# Composite influence score
df["influence_score"] = (
    0.5 * df["score_refs"] +
    0.3 * df["score_landmark"] +
    0.2 * df["score_so"]
)

top = df.nlargest(30, "influence_score")[
    ["number", "title", "status_clean", "year", "cited_by_count", "is_landmark", "influence_score"]
]
top.style.background_gradient(subset=["influence_score"], cmap="YlOrRd")

In [ ]:
# Bubble chart: most influential PEPs
top50 = df.nlargest(50, "influence_score").copy()
top50["label"] = top50.apply(lambda r: f"PEP {r['number']}", axis=1)

fig3 = px.scatter(
    top50,
    x="year",
    y="cited_by_count",
    size="influence_score",
    color="type",
    color_discrete_map=TYPE_COLORS,
    hover_name="label",
    hover_data={"title": True, "status_clean": True, "influence_score": ":.2f",
                "cited_by_count": True, "is_landmark": True},
    text="label",
    title="Top 50 Most Influential PEPs",
    labels={"year": "Year Created", "cited_by_count": "Cited by N other PEPs"},
    template="plotly_white",
    height=550,
    size_max=45,
)
fig3.update_traces(textposition="top center", textfont_size=9)
fig3.show()

## 4. How Python Has Changed Over Time

We look at the volume of influential PEPs per era, the dominant themes in each era, and which landmark PEPs defined each period.

In [ ]:
ERA_ORDER = [
    "Pre-2000",
    "Python 2 peak (2000–05)",
    "Py3 transition (2006–11)",
    "Stabilization (2012–15)",
    "Type hints era (2016–19)",
    "Modern Python (2020+)",
]

era_summary = df.groupby("era").agg(
    total_peps=("number", "count"),
    final_peps=("status_clean", lambda s: (s == "Final").sum()),
    rejected_peps=("status_clean", lambda s: (s == "Rejected").sum()),
    avg_influence=("influence_score", "mean"),
    landmark_count=("is_landmark", "sum"),
).reindex(ERA_ORDER).reset_index()

fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("PEPs per Era", "Avg Influence Score per Era"),
    horizontal_spacing=0.12,
)

fig4.add_trace(go.Bar(
    x=era_summary["era"], y=era_summary["total_peps"],
    name="Total PEPs", marker_color="#4C72B0",
), row=1, col=1)
fig4.add_trace(go.Bar(
    x=era_summary["era"], y=era_summary["final_peps"],
    name="Final PEPs", marker_color="#55A868",
), row=1, col=1)
fig4.add_trace(go.Bar(
    x=era_summary["era"], y=era_summary["rejected_peps"],
    name="Rejected PEPs", marker_color="#C44E52",
), row=1, col=1)

fig4.add_trace(go.Scatter(
    x=era_summary["era"], y=era_summary["avg_influence"],
    mode="lines+markers", name="Avg Influence",
    marker=dict(size=10, color="darkorange"),
    line=dict(color="darkorange", width=2),
), row=1, col=2)

fig4.update_layout(
    height=420, template="plotly_white",
    title="Python's PEP Activity and Influence by Era",
    barmode="group",
    xaxis=dict(tickangle=-30),
    xaxis2=dict(tickangle=-30),
    legend=dict(orientation="h", yanchor="bottom", y=-0.4),
)
fig4.show()

In [ ]:
# Landmark PEPs per era — annotated timeline
landmarks_df = df[df["is_landmark"] == 1].sort_values("created")

fig5 = go.Figure()

ERA_COLORS = {
    "Pre-2000": "#aaaaaa",
    "Python 2 peak (2000–05)": "#4C72B0",
    "Py3 transition (2006–11)": "#6baed6",
    "Stabilization (2012–15)": "#55A868",
    "Type hints era (2016–19)": "#fd8d3c",
    "Modern Python (2020+)": "#C44E52",
}

for era in ERA_ORDER:
    sub = landmarks_df[landmarks_df["era"] == era]
    fig5.add_trace(go.Scatter(
        x=sub["created"],
        y=[era] * len(sub),
        mode="markers+text",
        text=sub["number"].astype(str).map(lambda n: f"PEP {n}"),
        textposition="top center",
        textfont=dict(size=9),
        marker=dict(size=sub["influence_score"] * 60 + 8,
                    color=ERA_COLORS[era], opacity=0.85,
                    line=dict(width=1, color="white")),
        name=era,
        hovertext=sub.apply(
            lambda r: f"<b>PEP {r['number']}</b>: {r['title']}<br>"
                      f"Score: {r['influence_score']:.2f} | Cited by: {r['cited_by_count']}",
            axis=1,
        ),
        hoverinfo="text",
    ))

fig5.update_layout(
    title="Landmark PEPs Across Python's History<br><sub>Bubble size = influence score</sub>",
    xaxis_title="Date Created",
    yaxis=dict(categoryorder="array", categoryarray=list(reversed(ERA_ORDER)), title="Era"),
    height=500,
    template="plotly_white",
    showlegend=False,
    hovermode="closest",
)
fig5.show()

In [ ]:
# Era narrative: top 5 landmark PEPs per era with titles
print("=" * 70)
print("PYTHON'S EVOLUTION — TOP LANDMARK PEPs BY ERA")
print("=" * 70)

for era in ERA_ORDER:
    era_df = df[(df["era"] == era) & (df["is_landmark"] == 1)].nlargest(5, "influence_score")
    total = len(df[df["era"] == era])
    print(f"\n### {era} ({total} PEPs total) ###")
    if era_df.empty:
        print("  (no landmark PEPs)")
    for _, r in era_df.iterrows():
        star = "★" if r["is_landmark"] else " "
        print(f"  {star} PEP {r['number']:4d}  [{r['status_clean']:10s}]  {r['title']}")

## 5. PEP Reference Network (Top Nodes)

Which PEPs are most connected? We build an adjacency view of cross-references among the top 60 most-cited PEPs.

In [ ]:
top60_nums = set(df.nlargest(60, "cited_by_count")["number"])

# Build edge list: source → target (only within top-60 set)
edges = []
for _, row in df[df["number"].isin(top60_nums)].iterrows():
    for ref in row["refs"]:
        if ref in top60_nums and ref != row["number"]:
            edges.append((row["number"], ref))

# Simple heatmap adjacency matrix
top60_sorted = sorted(top60_nums)
label_map = {r["number"]: f"PEP {r['number']}" for _, r in df[df["number"].isin(top60_nums)].iterrows()}
idx = {n: i for i, n in enumerate(top60_sorted)}

import numpy as np
mat = np.zeros((len(top60_sorted), len(top60_sorted)), dtype=int)
for src, tgt in edges:
    mat[idx[src]][idx[tgt]] += 1

labels = [label_map[n] for n in top60_sorted]

fig6 = go.Figure(go.Heatmap(
    z=mat,
    x=labels,
    y=labels,
    colorscale="Blues",
    showscale=True,
    hovertemplate="<b>%{y}</b> references <b>%{x}</b><extra></extra>",
))
fig6.update_layout(
    title="Cross-Reference Heatmap — Top 60 Most-Cited PEPs<br><sub>Row → Col means row PEP references col PEP</sub>",
    height=750,
    xaxis=dict(tickfont=dict(size=8), tickangle=-90),
    yaxis=dict(tickfont=dict(size=8), autorange="reversed"),
    template="plotly_white",
)
fig6.show()